# パターン認識特論 第3回 演習：誤差逆伝播法とニューラルネットワークの訓練

スライドの「目的」と「方法」を，NumPy で動かして確かめる．導出（証明）は扱わない．
「何のために，何を，どの順で計算するか」と，「計算した勾配が正しいことをどう確かめるか」を体験する．

| 節 | テーマ | スライド | 
|---|---|---|
| §1 | 重み 1 つの数値例と学習率 | p.3–4 | 
| §2 | 計算例をたどる（2 層のネットワーク） | p.8 | 
| §3 | MLP の誤差逆伝播：手順 ①〜④ と，なぜ逆伝播か | p.5–7 | 
| §4 | SGD・Momentum・Adam | p.10–11 |
| §5 | 学習率とクリッピング | p.12 | 
| §6 | 勾配消失 | p.12 | 
| §7 | （任意）PyTorch の自動微分 | p.9 | 

**使い方**
- セルを選んで **Shift + Enter** で実行する．**必ず上から順に**実行する（下のセルは上のセルで作った変数や関数を使う）．
- 結果がおかしくなったら，メニューの「Kernel → Restart Kernel and Run All Cells」で最初からやり直す．
- **やってみよう** は，先に予想してから，`# TODO` の行を書き換えて実行する．`# TODO` がないものは，実行して結果を読む．
- 記号：訓練の 4 段階は **A〜D**（A 順伝播，B 損失，C 勾配を求める，D 更新する．p.3）．誤差逆伝播法の手順は **①〜④**（p.7）．①〜④ の全体が C の中身である．

環境：Python 3 ＋ NumPy ＋ Matplotlib．§7 のみ PyTorch（入っていなければ読み飛ばす）．

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)   # 数値を小数 4 桁で表示する
rng = np.random.default_rng(0)                    # 乱数（毎回同じ結果になるよう種を固定）

def sigmoid(u):
    return 1.0 / (1.0 + np.exp(-u))

def relu(u):
    return np.maximum(0.0, u)

## §1 重み 1 つの数値例と学習率（p.3–4）

訓練は「A 順伝播 → B 損失 → C 勾配を求める → D 更新する」の繰り返し（p.3）．
重み 1 つのモデル $\hat y = wx$，損失 $L=\frac{1}{2}(\hat y-y)^2$ で，これをそのまま書く（$x=2$，正解 $y=4$，最初の $w=1$）．
勾配は $\partial L/\partial w=(\hat y-y)\,x$．

In [ ]:
x, y = 2.0, 4.0

def run(eta, w=1.0, steps=8):
    # eta：学習率．steps 回の更新を行い，毎回の (回数, w, 損失, 勾配) を記録する
    hist = []
    for t in range(steps):
        y_hat = w * x                    # A 順伝播
        L = 0.5 * (y_hat - y) ** 2       # B 損失
        g = (y_hat - y) * x              # C 勾配
        hist.append((t, w, L, g))        # 記録：(回数, w, 損失, 勾配) の組
        w = w - eta * g                  # D 更新（勾配と逆の向きへ，学習率 × 勾配だけ動かす）
    return hist

for t, w, L, g in run(0.1):
    print(f"{t} 回目: w={w:6.3f}  損失 L={L:7.3f}  勾配={g:7.3f}")
# 0 回目は p.4 と同じ（w=1, L=2, 勾配=-4）．1 回目は w=1.4, L=0.72．

**やってみよう 1-1（p.4 のクイズ）**　学習率 $\eta$ を 1 にすると，1 回更新した後の $w$ と損失はいくらか．先に手で計算してから，下のセルを実行して確かめる（解答は末尾の表の p.4）．
次に，学習率を 0.5 と 0.25 に変えて実行する．

In [ ]:
for t, w, L, g in run(1.0, steps=3):      # TODO: 学習率を 0.5 や 0.25 にも変えてみる
    print(f"{t} 回目: w={w:8.3f}  損失 L={L:9.3f}")
# 学習率 0.25：1 回でちょうど損失が最小の w=2 に着く．
# 学習率 0.5 ：w が 1 と 3 を交互に動くだけで，損失は 2 のまま下がらない．0.5 を超えると，損失は更新のたびに増える．

**やってみよう 1-2（実行して結果を読む）**　学習率を変えて，損失が更新の回数とともにどう変わるかを描く．縦軸は対数目盛．
下がり続ける線，上がっていく線はそれぞれどの学習率か．

In [ ]:
plt.figure(figsize=(6, 3.6))
for eta in (0.1, 0.2, 0.3, 0.6):
    losses = [h[2] for h in run(eta, steps=8)]       # h は (回数, w, 損失, 勾配) の組．h[2] は 3 番目＝損失
    plt.semilogy(losses, marker='o', label=f'eta={eta}')   # semilogy：縦軸が対数目盛の plot．1 目盛り下がると 1/10
plt.xlabel('number of updates'); plt.ylabel('loss (log scale)'); plt.legend(); plt.grid(alpha=0.3); plt.show()
# 解答：0.1，0.2，0.3 は下がる（0.3 では，w が最小の位置 2 を毎回通り過ぎ，2 の上と下を交互に動きながら近づく）．
#       0.6 では，通り過ぎる量が毎回大きくなり，損失が増え続ける．

## §2 計算例をたどる（p.8）

p.8 の 2 層のネットワークを，誤差逆伝播法の手順 ①〜④ のとおりに計算する．

$x \;\xrightarrow{\times w_1}\; u_1 \;\xrightarrow{\text{ReLU}}\; z_1 \;\xrightarrow{\times w_2}\; u_2 \;\xrightarrow{\text{シグモイド}}\; p \;\rightarrow\; \text{損失 } L$

$x=1$，$w_1=0.5$，$w_2=-1$，正解 $y=1$．損失は二値交差エントロピー（正解が 1 なので $L=-\log p$）．

In [ ]:
def backprop_example(w1, w2, x=1.0, y=1.0):
    # ① 順伝播（途中の値を覚えておく）
    u1 = w1 * x
    z1 = relu(u1)
    u2 = w2 * z1
    p = sigmoid(u2)
    L = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    # ② 出力層の誤差信号 ＝ 予測 − 正解
    d2 = p - y
    # ③ 逆伝播：次の層の誤差信号に，同じ重み w2 と，自分の活性化関数（ReLU）の傾きを掛ける
    slope = 1.0 if u1 > 0 else 0.0       # ReLU の傾き：u1 > 0 なら 1，そうでなければ 0
    d1 = slope * w2 * d2
    # ④ 重みの勾配 ＝ 誤差信号 × 入力側の出力
    g_w2 = d2 * z1
    g_w1 = d1 * x
    # 結果を名前つきでまとめて返す．r['p'] のように名前で取り出せる
    return dict(u1=u1, z1=z1, u2=u2, p=p, L=L, d2=d2, d1=d1, g_w1=g_w1, g_w2=g_w2)

r = backprop_example(0.5, -1.0)
for name, value in r.items():
    print(f"{name:5s} = {value: .4f}")
# p.8 と同じ：p=0.378, d2=-0.622, d1=0.622, g_w2=-0.311, g_w1=0.622

### 勾配が正しいことを確かめる：数値微分

勾配は「その重みを少し増やしたとき，損失がどれだけ増えるか」．それなら，**実際に少し動かして損失の変化を測れば**，同じ値が出るはず．
この確かめ方を **数値微分** という（p.5 の「1 個ずつ調べる」方法）．

In [ ]:
eps = 1e-6                                # 動かす幅
# (増やしたときの損失 − 減らしたときの損失) ÷ (動かした幅 2×eps)．増やす・減らすの両方を測ると精度がよい

def loss_only(w1, w2):
    return backprop_example(w1, w2)['L']

num_g_w1 = (loss_only(0.5 + eps, -1.0) - loss_only(0.5 - eps, -1.0)) / (2 * eps)   # w1 だけ少し動かす
num_g_w2 = (loss_only(0.5, -1.0 + eps) - loss_only(0.5, -1.0 - eps)) / (2 * eps)   # w2 だけ少し動かす
print(f"逆伝播  : g_w1 = {r['g_w1']:.6f}, g_w2 = {r['g_w2']:.6f}")
print(f"数値微分: g_w1 = {num_g_w1:.6f}, g_w2 = {num_g_w2:.6f}")   # 一致する

**やってみよう 2-1**　求めた勾配で 1 回更新する（学習率 0.5）．更新後の $p$ と損失は，p.8 のポイントの値（$p$：0.38 → 0.46）になるか．

In [ ]:
eta = 0.5
w1_new = 0.5 - eta * r['g_w1']
w2_new = -1.0 - eta * r['g_w2']
r_new = backprop_example(w1_new, w2_new)
print(f"更新後: w1 = {w1_new:.3f}, w2 = {w2_new:.3f}")
print(f"p: {r['p']:.3f} → {r_new['p']:.3f},  損失: {r['L']:.3f} → {r_new['L']:.3f}")

**やってみよう 2-2**　$w_1=-0.5$ から始めると，誤差信号と勾配はどうなるか．先に予想してから実行する（ヒント：$u_1<0$ のとき ReLU の傾きは 0．p.6 のクイズ）．

In [ ]:
r_neg = backprop_example(0.5, -1.0)      # TODO: 0.5 を -0.5 に変えて実行する
print({k: round(float(v), 4) for k, v in r_neg.items()})

In [ ]:
# 解答
r_neg = backprop_example(-0.5, -1.0)
print("d1 =", r_neg['d1'], " g_w1 =", r_neg['g_w1'], " g_w2 =", r_neg['g_w2'])
# u1 < 0 なので z1 = 0，ReLU の傾きも 0．誤差信号 d1 は 0 になり，両方の勾配が 0．
# 勾配が 0 なので，何回更新しても w1 も w2 も動かない（第2回 p.9 の dying ReLU）．

## §3 MLP の誤差逆伝播：手順 ①〜④ と，なぜ逆伝播か（p.5–7）

ユニットが複数ある普通の MLP で，手順 ①〜④ を行列の式のまま書く（p.6–7）．

| 手順 | 式（スライド） | コード（下の `mlp_backward`） |
|---|---|---|
| ① 順伝播 | $\boldsymbol{u}^{(l)} = W^{(l)}\boldsymbol{z}^{(l-1)} + \boldsymbol{b}^{(l)}$，$\boldsymbol{z}^{(l)} = h(\boldsymbol{u}^{(l)})$ | `mlp_forward` |
| ② 出力層の誤差信号 | 出力層の $\boldsymbol{\delta} = \boldsymbol{p} - \boldsymbol{y}$ | `D = (Pr - Y) / N` |
| ③ 逆伝播 | $\boldsymbol{\delta}^{(l)} = h'(\boldsymbol{u}^{(l)}) \odot (W^{(l+1)\top}\boldsymbol{\delta}^{(l+1)})$ | `D = (D @ P[f'W{l}']) * (U_prev > 0)` |
| ④ 重みの勾配 | $\partial L/\partial W^{(l)} = \boldsymbol{\delta}^{(l)}\boldsymbol{z}^{(l-1)\top}$ | `G[f'W{l}'] = D.T @ Z_prev` |

コードの `D` は誤差信号．複数のデータを一度に扱い，1 行が 1 件のデータなので，式の $W^\top\boldsymbol{\delta}$ は `D @ W`，$\boldsymbol{\delta}\boldsymbol{z}^\top$ は `D.T @ Z` と書ける（転置の位置が式と違って見えるが，同じ計算）．
ループは出力側の層から順に回る．各層で，先に ④（その層の重みの勾配）を計算し，次に ③ で `D` を 1 つ入力側の層の誤差信号に作り替える．
**読むのは `mlp_backward` の中の，②③④ と書いた 3 行だけ**でよい．

In [ ]:
# MLP（隠れ層 ReLU，出力層ソフトマックス，損失は交差エントロピー）
def softmax_rows(U):
    U = U - U.max(axis=1, keepdims=True)
    E = np.exp(U)
    return E / E.sum(axis=1, keepdims=True)

def init_mlp(sizes, seed=0):
    # sizes：各層のユニット数．例 [2, 16, 2] は 入力 2 → 隠れ 16 → 出力 2
    r = np.random.default_rng(seed)
    P = {}
    for l in range(1, len(sizes)):
        P[f'W{l}'] = r.normal(size=(sizes[l], sizes[l - 1])) / np.sqrt(sizes[l - 1])
        P[f'b{l}'] = np.zeros(sizes[l])
    return P

def n_params(P):
    return sum(v.size for v in P.values())

forward_count = 0                          # 順伝播を何回行ったかを数える（§3 後半で使う）

def mlp_forward(P, X):
    # ① 順伝播．各層の総入力 U と出力 Z を cache に覚えておく
    global forward_count
    forward_count += 1
    n_layers = len(P) // 2
    cache = [(None, X)]
    Z = X
    for l in range(1, n_layers + 1):
        U = Z @ P[f'W{l}'].T + P[f'b{l}']
        Z = relu(U) if l < n_layers else U       # 最後の層は活性化関数に通す前の値（ロジット）
        cache.append((U, Z))
    return Z, cache

def mlp_loss(P, X, y):
    logits, _ = mlp_forward(P, X)
    Pr = softmax_rows(logits)
    return -np.mean(np.log(Pr[np.arange(len(y)), y] + 1e-12))

def mlp_backward(P, X, y):
    n_layers = len(P) // 2
    logits, cache = mlp_forward(P, X)             # ① 順伝播
    N = len(y)
    Pr = softmax_rows(logits)                     # 予測（確率）
    Y = np.eye(logits.shape[1])[y]                # 正解（1-hot）
    D = (Pr - Y) / N                              # ② 出力層の誤差信号 ＝ 予測 − 正解（N 件の平均にするため N で割る）
    G = {}
    for l in range(n_layers, 0, -1):              # 出力側の層から順に．ここでの D は第 l 層の誤差信号
        U_prev, Z_prev = cache[l - 1]             # 1 つ入力側（第 l−1 層）の総入力と出力
        G[f'W{l}'] = D.T @ Z_prev                 # ④ 重みの勾配 ＝ 誤差信号 × 入力側の出力
        G[f'b{l}'] = D.sum(axis=0)                #    バイアスの勾配 ＝ 誤差信号（データの分だけ足す）
        if l > 1:
            D = (D @ P[f'W{l}']) * (U_prev > 0)   # ③ 逆伝播：同じ重みで集め，第 l−1 層の ReLU の傾き（1 か 0）を掛ける → 第 l−1 層の誤差信号
    return G

### 逆伝播で求めた勾配は正しいか（勾配チェック）

§2 と同じく，数値微分（パラメータを 1 個ずつ少し動かす）と比べる．小さなネットワーク（入力 3 → 隠れ 4 → 出力 3，パラメータ 31 個）で行う．

In [ ]:
def numerical_grad(P, X, y, eps=1e-5):
    # 全パラメータを 1 個ずつ動かして勾配を測る（数値微分）
    G = {}
    for k in P:
        G[k] = np.zeros_like(P[k])
        for idx in np.ndindex(P[k].shape):        # このパラメータ行列の全成分を順に
            old = P[k][idx]
            P[k][idx] = old + eps; L_plus = mlp_loss(P, X, y)
            P[k][idx] = old - eps; L_minus = mlp_loss(P, X, y)
            P[k][idx] = old
            G[k][idx] = (L_plus - L_minus) / (2 * eps)
    return G

X_small = rng.normal(size=(5, 3))                 # データ 5 件，入力 3 次元
y_small = np.array([0, 2, 1, 2, 0])               # 正解クラス
P_small = init_mlp([3, 4, 3])
G_bp = mlp_backward(P_small, X_small, y_small)    # 逆伝播
G_num = numerical_grad(P_small, X_small, y_small) # 数値微分
for k in P_small:
    print(f"{k}: 2 つの勾配の差の最大値 = {np.abs(G_bp[k] - G_num[k]).max():.2e}")
# どれも 1e-8 程度以下なら，逆伝播の計算は正しい

**やってみよう 3-1（実行して結果を読む）：なぜ誤差逆伝播法か（p.5）**

同じ勾配が出るなら，なぜ数値微分を使わないのか．必要な順伝播の回数と時間を比べる．
ネットワークを少し大きくする（入力 20 → 隠れ 50 → 隠れ 50 → 出力 3）．

In [ ]:
X_big = rng.normal(size=(100, 20)); y_big = rng.integers(0, 3, size=100)
P_big = init_mlp([20, 50, 50, 3])
print("パラメータ数 =", n_params(P_big))

# forward_count は，mlp_forward が呼ばれるたびに 1 増えるカウンタ．0 に戻してから測る
# time.perf_counter() は現在の時刻（秒）．前後の差が，かかった時間
forward_count = 0
t0 = time.perf_counter()
G1 = mlp_backward(P_big, X_big, y_big)
t_bp = time.perf_counter() - t0
print(f"誤差逆伝播法: 順伝播 {forward_count} 回 ＋ 逆伝播 1 回，{t_bp * 1000:.1f} ミリ秒")

forward_count = 0
t0 = time.perf_counter()
G2 = numerical_grad(P_big, X_big, y_big)
t_num = time.perf_counter() - t0
print(f"数値微分    : 順伝播 {forward_count} 回，{t_num:.1f} 秒（逆伝播の約 {t_num / t_bp:.0f} 倍）")
print("2 つの勾配の差の最大値 =", f"{max(np.abs(G1[k] - G2[k]).max() for k in P_big):.2e}")
# 解答：勾配は同じだが，数値微分は順伝播がパラメータ数の 2 倍必要．
#       （スライド p.5 の「P+1 回」は増やすだけの場合．ここでは精度のため増やす・減らすの両方を測るので 2P 回．どちらも P に比例する）
#       パラメータが 1 兆個なら，この差は 1 兆倍の規模になる．数値微分は「検算」にだけ使う．

### 訓練する：A → B → C → D の繰り返し（p.3）

同心円のデータ（第2回 §3 と同じ）を学習する．C（勾配を求める）に上の `mlp_backward` を使い，D（更新する）は p.10 の式 $\boldsymbol{\theta}\leftarrow\boldsymbol{\theta}-\eta\boldsymbol{g}$ のとおりに書く．
データが 400 点と少ないので，ここでは毎回全部のデータで勾配を求める（データが多いときは，毎回一部を無作為に選ぶ．それが p.10 のミニバッチ）．

In [ ]:
# データを作る（読まなくてよい）．内側の円（クラス 0）と外側の輪（クラス 1）．各 200 点
n = 200
radius = np.concatenate([rng.uniform(0.0, 1.0, n), rng.uniform(1.6, 2.6, n)])
angle = rng.uniform(0, 2 * np.pi, 2 * n)
X_c = np.stack([radius * np.cos(angle), radius * np.sin(angle)], axis=1)
y_c = np.concatenate([np.zeros(n, int), np.ones(n, int)])

P = init_mlp([2, 16, 2])
eta = 0.5                                  # 学習率  TODO: 0.05 や 6.0 に変えるとどうなるか（解答は下）
losses = []
for step in range(2000):
    G = mlp_backward(P, X_c, y_c)          # A・B・C：誤差逆伝播法の手順 ①〜④ で勾配を求める
    for k in P:
        P[k] -= eta * G[k]                 # D：更新する
    losses.append(mlp_loss(P, X_c, y_c))
logits, _ = mlp_forward(P, X_c)
print("正解率 =", np.mean(logits.argmax(axis=1) == y_c))
plt.figure(figsize=(6, 3.2)); plt.plot(losses); plt.xlabel('number of updates'); plt.ylabel('loss'); plt.grid(alpha=0.3); plt.show()
# 解答：0.05 では損失は下がるが遅い．6.0 では損失が増えたり減ったりして下がらない（p.4 のクイズと同じ現象）．

## §4 SGD・Momentum・Adam（p.10–11）

p.10 と同じ損失 $L=(w_1^2+12w_2^2)/2$ の上で，3 つの更新方法がパラメータ $(w_1, w_2)$ をどう動かすかを描く（SGD と Momentum は p.10 と同じ設定：学習率 0.16，30 回）．
最小は中心 $(0,0)$．$w_2$ の方向は傾きが急（係数 12），$w_1$ の方向はゆるやか．勾配は $(w_1,\ 12w_2)$．
下のセルの前半は 3 つの更新方法の定義（スライドの式のとおり．Adam だけ補正の 1 行が加わる），後半は描画．

In [ ]:
coef = np.array([1.0, 12.0])                 # 各方向の係数（w1 は 1，w2 は 12）

def grad(w):
    return coef * w                          # 損失 (w1² + 12 w2²)/2 の勾配 (w1, 12 w2)

def sgd(w0, eta, steps):
    w = np.array(w0, float); path = [w.copy()]
    for _ in range(steps):
        w = w - eta * grad(w)                # θ ← θ − η g
        path.append(w.copy())
    return np.array(path)

def momentum(w0, eta, alpha, steps):
    w = np.array(w0, float); v = np.zeros(2); path = [w.copy()]
    for _ in range(steps):
        v = alpha * v - eta * grad(w)        # v ← α v − η g
        w = w + v                            # θ ← θ + v
        path.append(w.copy())
    return np.array(path)

def adam(w0, eta, steps, b1=0.9, b2=0.999, eps=1e-8):
    w = np.array(w0, float); m = np.zeros(2); v = np.zeros(2); path = [w.copy()]
    for t in range(1, steps + 1):
        g = grad(w)
        m = b1 * m + (1 - b1) * g            # 勾配の平均
        v = b2 * v + (1 - b2) * g ** 2       # 勾配の 2 乗の平均
        m_hat = m / (1 - b1 ** t); v_hat = v / (1 - b2 ** t)   # 補正：m, v は 0 から始めるので初めは小さく出る．それを直す（スライドでは省略）
        w = w - eta * m_hat / (np.sqrt(v_hat) + eps)
        path.append(w.copy())
    return np.array(path)

def final_loss(path):
    return 0.5 * (path[-1, 0] ** 2 + 12 * path[-1, 1] ** 2)

w0 = (4.0, 1.5)                              # 出発点
n_updates = 30
paths = {'SGD (eta=0.16)': sgd(w0, 0.16, n_updates),
         'Momentum (eta=0.16, alpha=0.5)': momentum(w0, 0.16, 0.5, n_updates),
         'Adam (eta=0.3)': adam(w0, 0.3, n_updates)}   # Adam の学習率は，動きが見えるよう大きくしてある（実際の訓練では 0.001 程度）

g1, g2 = np.meshgrid(np.linspace(-2.5, 4.5, 200), np.linspace(-2, 2, 200))
L_grid = 0.5 * (g1 ** 2 + 12 * g2 ** 2)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (name, path) in zip(axes, paths.items()):
    ax.contour(g1, g2, L_grid, levels=[0.25, 0.5, 1, 2, 4, 8], colors='gray', linewidths=0.6)   # 損失が等しい線
    ax.plot(path[:, 0], path[:, 1], '-o', ms=3)
    ax.plot(path[-1, 0], path[-1, 1], 'ks', ms=6)                                               # 30 回目の位置
    ax.set_title(f"{name}\nloss after {n_updates} updates = {final_loss(path):.2e}")
    ax.set_aspect('equal'); ax.set_xlabel('w1'); ax.set_ylabel('w2')
plt.tight_layout(); plt.show()

**図の見方**　灰色の楕円は損失が等しい線（中心が最小）．点は 1 回ごとの位置，■ は 30 回目．題の数値は 30 回後の損失．
- SGD：上下（$w_2$）の向きが毎回反転し，振動がなかなか小さくならない．
- Momentum：上下の動きが互いに打ち消され，同じ学習率でも 30 回後の損失がはるかに小さい．
- Adam：勾配の大きい $w_2$ も小さい $w_1$ も，1 回に動く量がおよそ学習率（0.3）でそろう．量が一定なので，最小の近くでは通り過ぎて周りを動く．
  この図で見るのは「動く量がそろうこと」で，損失の小ささの比較ではない．Adam の利点は，学習率の値を決めやすいこと（やってみよう 4-2）．

**やってみよう 4-1**　Momentum の $\alpha$ を 0，0.5，0.7，0.9 と変えると，30 回後の損失はどうなるか．大きいほど良いか．先に予想してから実行する．

In [ ]:
for alpha in (0.0, 0.5, 0.7, 0.9):       # TODO: 別の値も試す
    print(f"alpha={alpha}: 30 回後の損失 = {final_loss(momentum(w0, 0.16, alpha, n_updates)):.2e}")
# 解答：この例では 0.5 が最も小さい．0.9 では速度 v が大きくなりすぎて最小の位置を通り過ぎ，30 回後の損失は α=0 より大きい．
#       α は「大きいほど良い」わけではない．何千回も更新する実際の訓練では 0.9 がよく使われるが，適切な値は問題による．

**やってみよう 4-2（実行して結果を読む）**　SGD の学習率を 0.16 から 0.17 に少し上げると，$w_2$ の方向で発散する（最小から離れていく）．同じ 0.17 でも Adam は発散しない．なぜか（p.11）．

In [ ]:
p_sgd = sgd(w0, 0.17, 40); p_adam = adam(w0, 0.17, 40)
print("SGD  (eta=0.17): 40 回後の |w2| =", round(abs(p_sgd[-1, 1]), 2), "（出発点は 1.5．最小の位置 0 から離れていく）")
print("Adam (eta=0.17): 40 回後の損失 =", round(final_loss(p_adam), 3), "（出発点の損失は 21.5）")
# 解答：SGD は「学習率 × 勾配」だけ動くので，勾配が大きい w2 の方向で動きすぎる．
#       Adam は勾配をその大きさで割るので，1 回に動く量がおよそ学習率にそろい，動きすぎない．

## §5 学習率とクリッピング（p.12）

### 5.1 学習率の大小で，損失の下がり方はどう変わるか

§3 の同心円データと MLP を，学習率だけ変えて訓練する．**先に予想する**：小さすぎる学習率，大きすぎる学習率では，損失の線はそれぞれどうなるか．

In [ ]:
def train(eta_fn, steps=1500, clip=None, spike=False):
    # 同じ初期値から訓練し，損失の記録を返す．eta_fn(t) は t 回目の学習率（lambda t: 0.5 は「いつも 0.5」という関数）
    P = init_mlp([2, 16, 2], seed=1)
    hist = []
    for t in range(steps):
        G = mlp_backward(P, X_c, y_c)
        if spike and t == 49:                              # 5.3 用：50 回目の更新で 1 回だけ，勾配を 1000 倍にする
            G = {k: g * 1000.0 for k, g in G.items()}
        if clip is not None:                               # 勾配クリッピング（p.13 の式）
            norm = np.sqrt(sum((g ** 2).sum() for g in G.values()))   # 全パラメータの勾配をまとめた長さ
            factor = min(1.0, clip / (norm + 1e-12))
            G = {k: g * factor for k, g in G.items()}
        eta = eta_fn(t)
        for k in P:
            P[k] -= eta * G[k]
        hist.append(mlp_loss(P, X_c, y_c))
    return np.array(hist)

settings = {'eta=0.02 (too small)': lambda t: 0.02,
            'eta=0.5': lambda t: 0.5,
            'eta=6.0 (too large)': lambda t: 6.0}
plt.figure(figsize=(7, 3.8))
for name, eta_fn in settings.items():
    h = train(eta_fn)
    plt.plot(h, label=f"{name}  final loss={h[-1]:.3f}")
plt.ylim(0, 1.0); plt.xlabel('number of updates'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3); plt.show()
# 解答：小さすぎる（0.02）と下がるのが遅い．大きすぎる（6.0）と，損失が大きく増えたり減ったりして下がらない．
#       学習の初めに大きな学習率を使うことが問題で，少し進んでからなら大きな学習率を使える．

### 5.3 勾配クリッピング（p.12）

$\boldsymbol{g}\leftarrow \boldsymbol{g}\times\min(1,\ c/\|\boldsymbol{g}\|)$．まずスライドの例を計算する．
次に，「極端に大きな勾配が出る」状況を人工的に作り（50 回目の更新で 1 回だけ，勾配を 1000 倍にする），クリッピングの有無で訓練を比べる．

In [ ]:
def clip_by_norm(g, c):
    return g * min(1.0, c / np.linalg.norm(g))       # np.linalg.norm：ベクトルの長さ

print("g=(3,4),     c=2.5 →", clip_by_norm(np.array([3.0, 4.0]), 2.5))    # 長さ 5 → 0.5 倍して (1.5, 2.0)
print("g=(0.6,0.8), c=2.5 →", clip_by_norm(np.array([0.6, 0.8]), 2.5))    # 長さ 1 は上限以下 → そのまま

plt.figure(figsize=(7, 3.6))
for clip in (None, 1.0):
    h = train(lambda t: 0.5, steps=200, clip=clip, spike=True)
    plt.plot(h, label=f"clip={clip}")
    print(f"clip={clip}: 49 回目の損失 {h[48]:.3f} → 50 回目の損失 {h[49]:.3f}")
plt.xlabel('number of updates'); plt.ylabel('loss'); plt.legend(); plt.grid(alpha=0.3); plt.show()
# 読み取り：クリッピングなし（clip=None）では，50 回目に損失が急に大きくなる．クリッピングあり（clip=1.0）では大きくならない．
#         クリッピングは，極端に大きな勾配が出たときに，1 回の更新が大きくなりすぎることを防ぐ．

## §6 勾配消失（p.12）

逆伝播では，誤差信号に各層の傾き $h'$ と重み $W^\top$ が次々に掛かる（p.6）．層が多いと，入力側に届くころには極端に小さくなることがある．
20 層のネットワーク（各層 64 ユニット）で，シグモイドの場合と ReLU の場合に，各層の誤差信号の大きさを測る．
学習はしない．重みは乱数のまま，順伝播を 1 回，逆伝播を 1 回行うだけである．

In [ ]:
def delta_sizes(act, n_layers=20, width=64, scale=1.0, seed=0):
    # 各層の誤差信号の大きさ（入力側 → 出力側の順）を返す
    r = np.random.default_rng(seed)
    X = r.normal(size=(32, width))
    Ws, Us, Z = [], [], X
    for l in range(n_layers):                                # 順伝播
        W = r.normal(size=(width, width)) * scale / np.sqrt(width)
        U = Z @ W.T
        Z = sigmoid(U) if act == 'sigmoid' else relu(U)
        Ws.append(W); Us.append(U)
    D = r.normal(size=Z.shape) / np.sqrt(width)              # 出力層の誤差信号（ここでは乱数で代用する）
    sizes = []
    for l in range(n_layers - 1, -1, -1):                    # 逆伝播
        slope = sigmoid(Us[l]) * (1 - sigmoid(Us[l])) if act == 'sigmoid' else (Us[l] > 0)
        D = D * slope                                        # 傾き h' を掛ける
        sizes.append(np.linalg.norm(D) / np.sqrt(D.shape[0]))
        D = D @ Ws[l]                                        # 同じ重みで入力側へ集める
    return np.array(sizes[::-1])

plt.figure(figsize=(6, 3.6))
for act, scale in (('sigmoid', 1.0), ('relu', np.sqrt(2.0))):
    plt.semilogy(range(1, 21), delta_sizes(act, scale=scale), '-o', ms=3, label=act)
plt.xlabel('layer (1 = input side, 20 = output side)'); plt.ylabel('size of error signal (log scale)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

**読み取り**　右端（出力側）から左（入力側）へ逆伝播する．シグモイドでは，1 層戻るごとに誤差信号が小さくなり，入力側では何桁も小さい（勾配消失）．
ReLU ではほぼ同じ大きさのまま届く．`scale` は重みの初期値の大きさを決める数で，ReLU では $\sqrt{2}$ にしてある（理由は第6回）．

**やってみよう 6-1（実行して結果を読む）**　ReLU でも，重みの初期値を小さくする（`scale=1.0`）と消失するか．初期値の決め方がなぜ重要かを考える（第6回）．

In [ ]:
for act, scale in (('sigmoid', 1.0), ('relu', 1.0), ('relu', np.sqrt(2.0))):
    s = delta_sizes(act, scale=scale)
    print(f"{act:8s} scale={scale:.2f}: 入力側の層 ÷ 出力側の層 = {s[0] / s[-1]:.2e}")
# 解答：ReLU でも scale=1.0 では約 1/1000 に小さくなる．√2 倍にすると，ほぼ同じ大きさのまま届く．

## §7（任意）PyTorch の自動微分（p.9）

PyTorch では，順伝播を書いて `loss.backward()` を呼ぶと，手順 ②③④ が自動で実行される．PyTorch が入っていなければ，この節は読み飛ばす．

| 項 | 内容 | 対応する節 |
|---|---|---|
| 7.1 | 計算例（p.8）の勾配を `loss.backward()` で求め，手計算と比べる | §2 |
| 7.2 | 訓練する：順伝播 → `loss.backward()` → `optimizer.step()` | §3 |
| 7.3 | 勾配が計算されなくなる例 | p.9 |

In [ ]:
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    print("PyTorch", torch.__version__)
except ImportError:
    HAS_TORCH = False
    print("PyTorch が見つからないので，§7 のセルは何もしない（読み飛ばす）．")

### 7.1 計算例（p.8）を自動微分で

`requires_grad=True` は「この値の勾配を求める」という指定．順伝播だけを書く．

In [ ]:
if HAS_TORCH:
    w1 = torch.tensor(0.5, requires_grad=True)
    w2 = torch.tensor(-1.0, requires_grad=True)
    x_t, y_t = torch.tensor(1.0), torch.tensor(1.0)

    u1 = w1 * x_t                          # ① 順伝播（§2 と同じ計算）
    z1 = torch.relu(u1)
    u2 = w2 * z1
    p = torch.sigmoid(u2)
    loss = -(y_t * torch.log(p) + (1 - y_t) * torch.log(1 - p))

    loss.backward()                        # ②③④ を自動で実行
    print(f"p = {p.item():.4f}, 損失 = {loss.item():.4f}")
    print(f"自動微分: g_w1 = {w1.grad.item():.4f}, g_w2 = {w2.grad.item():.4f}")
    print(f"§2 の手順: g_w1 = {r['g_w1']:.4f}, g_w2 = {r['g_w2']:.4f}")     # 一致する（0.6225, -0.3112）

### 7.2 訓練する

§3 の訓練（同心円データ）を PyTorch で書く．`mlp_backward` に当たる部分が `loss.backward()` の 1 行になる．
更新は `torch.optim.SGD`（p.10）．`Adam` に変えることもできる（やってみよう）．

In [ ]:
if HAS_TORCH:
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))   # 入力 2 → 隠れ 16 → 出力 2
    X_t = torch.tensor(X_c, dtype=torch.float32)
    y_tc = torch.tensor(y_c, dtype=torch.long)
    loss_fn = nn.CrossEntropyLoss()                                         # ソフトマックス＋交差エントロピー
    optimizer = torch.optim.SGD(model.parameters(), lr=0.5)                 # TODO: torch.optim.Adam(model.parameters(), lr=0.01) に変える
    losses_t = []
    for step in range(2000):
        loss = loss_fn(model(X_t), y_tc)   # ① 順伝播と ② 損失
        optimizer.zero_grad()              # 前回の勾配を消す
        loss.backward()                    # ②③④ 誤差信号と勾配（自動）
        optimizer.step()                   # 更新
        losses_t.append(loss.item())
    with torch.no_grad():
        acc = (model(X_t).argmax(dim=1) == y_tc).float().mean().item()
    print("正解率 =", acc)
    plt.figure(figsize=(6, 3.2)); plt.plot(losses_t); plt.xlabel('number of updates'); plt.ylabel('loss'); plt.grid(alpha=0.3); plt.show()

### 7.3 勾配が計算されなくなる例（p.9）

順伝播の途中の値を `.detach()` で取り出し，その値で続きを計算すると，それより入力側のパラメータには勾配が計算されない．エラーは出ない．

In [ ]:
if HAS_TORCH:
    w1 = torch.tensor(0.5, requires_grad=True)
    w2 = torch.tensor(-1.0, requires_grad=True)
    z1 = torch.relu(w1 * torch.tensor(1.0))
    z1_cut = z1.detach()                   # ← 値だけを取り出す．ここより入力側（w1）へは勾配が計算されない
    p = torch.sigmoid(w2 * z1_cut)
    loss = -torch.log(p)
    loss.backward()
    print("g_w2 =", w2.grad.item(), "（届く）")
    print("g_w1 =", w1.grad, "（None：勾配が計算されていない．w1 は何回訓練しても更新されない）")